# Pix2Pix Maps Training on Colab

This notebook clones the repo, trains the model for 200 epochs, and runs inference.

**Before running:**
1. Enable GPU: Runtime > Change runtime type > T4 GPU

In [ ]:
# 1. Mount Google Drive to save checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone the repo
!git clone https://github.com/5cisummai/UNET-pix2pix-maps.git /content/UNET-pix2pix-maps
%cd /content/UNET-pix2pix-maps

In [ ]:
# 3. Install dependencies (single GPU, no DDP)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
!pip install numpy Pillow PyYAML tensorboard tqdm

In [ ]:
# 4. Download the maps dataset
!python download_pix2pix_maps.py

In [ ]:
# 5. Train for 200 epochs, save checkpoints to Google Drive
!python train_pix2pix.py \
  --data-root datasets/maps \
  --output-dir /content/drive/MyDrive/pix2pix_runs \
  --epochs 200 \
  --generator-norm instance \
  --discriminator-norm instance \
  --target-interpolation nearest \
  --generator-lr 2e-4 \
  --discriminator-lr 1e-4 \
  --lambda-l1 100 \
  --d-update-interval 2 \
  --patience 40 \
  --no-use-ddp

In [ ]:
# 6. Run inference on validation set using the best checkpoint
!python infer_pix2pix.py \
  --checkpoint /content/drive/MyDrive/pix2pix_runs/checkpoints/best.pt \
  --input-path datasets/maps/val \
  --output-dir /content/drive/MyDrive/pix2pix_inference \
  --paired-input \
  --source-side left

In [ ]:
# 7. Show a few generated outputs
from IPython.display import display, Image as IPImage
from pathlib import Path

output_dir = Path('/content/drive/MyDrive/pix2pix_inference')
for img_path in sorted(output_dir.glob('*_generated.png'))[:5]:
    print(img_path.name)
    display(IPImage(filename=str(img_path), width=512))
    print()

## Resume Training Later

If Colab disconnects, you can resume from the saved checkpoint:

```python
!python train_pix2pix.py \
  --data-root datasets/maps \
  --output-dir /content/drive/MyDrive/pix2pix_runs \
  --epochs 200 \
  --resume /content/drive/MyDrive/pix2pix_runs/checkpoints/latest.pt \
  --generator-norm instance \
  --discriminator-norm instance \
  --target-interpolation nearest \
  --generator-lr 2e-4 \
  --discriminator-lr 1e-4 \
  --lambda-l1 100 \
  --d-update-interval 2 \
  --patience 40 \
  --no-use-ddp
```